# SASRec BPI2012 Colab Sanity Check (Eval Fix)

Colab notebook for sanity checking the two selected post-fix baseline candidates.

Goals:
- reuse already completed runs instead of retraining them
- train only the missing seeds for the two selected candidate settings
- summarize mean/std and valid-test trends separately for `NDCG@10` and `NDCG@5` model-selection criteria


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('NDCG10_OUTPUT_DIR:', NDCG10_OUTPUT_DIR)
print('NDCG5_OUTPUT_DIR:', NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$NDCG10_OUTPUT_DIR"
!mkdir -p "$NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Candidate settings

Selected candidates for sanity check:
- `anchor_ml20` (`hidden_units=32, maxlen=20, dropout=0.2`)
- `refine_ml50_do035` (`hidden_units=50, maxlen=50, dropout=0.35`)

We will evaluate them under two model-selection criteria separately:
- `full_valid_ndcg@10`
- `full_valid_ndcg@5`


## Check existing completed runs

These runs should already exist and **must not be retrained**.


In [10]:
from pathlib import Path

existing_ndcg10 = [
    'anchor_ml20_s42',
    'refine_ml50_do035_s42',
]
existing_ndcg5 = [
    'anchor_ml20_s42',
    'refine_ml50_do035_s42',
]

for label, output_dir, run_names in [
    ('NDCG@10', Path(NDCG10_OUTPUT_DIR), existing_ndcg10),
    ('NDCG@5', Path(NDCG5_OUTPUT_DIR), existing_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


NDCG@10
anchor_ml20_s42 EXISTS
refine_ml50_do035_s42 EXISTS
NDCG@5
anchor_ml20_s42 EXISTS
refine_ml50_do035_s42 EXISTS


## Train only missing seeds for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### anchor_ml20_s2024


In [11]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/anchor_ml20_s2024
epoch=1, loss=0.6261
epoch=2, loss=0.3053
epoch=3, loss=0.2087
epoch=4, loss=0.1653
epoch=5, loss=0.1432
valid [full], NDCG@5: 0.6528, HR@5: 0.7732, NDCG@10: 0.7159, HR@10: 0.9707, MRR: 0.6426
valid [sampled], NDCG@5: 0.5588, HR@5: 0.5593, NDCG@10: 0.5623, HR@10: 0.5710, MRR: 0.5754
test [full], NDCG@5: 0.7080, HR@5: 0.8769, NDCG@10: 0.7503, HR@10: 1.0000, MRR: 0.6720
test [sampled], NDCG@5: 0.2013, HR@5: 0.2642, NDCG@10: 0.2488, HR@10: 0.4097, MRR: 0.2276
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outp

### anchor_ml20_s7


In [12]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/anchor_ml20_s7
epoch=1, loss=0.6517
epoch=2, loss=0.3093
epoch=3, loss=0.2163
epoch=4, loss=0.1750
epoch=5, loss=0.1489
valid [full], NDCG@5: 0.6739, HR@5: 0.8165, NDCG@10: 0.7175, HR@10: 0.9552, MRR: 0.6484
valid [sampled], NDCG@5: 0.5554, HR@5: 0.5575, NDCG@10: 0.5649, HR@10: 0.5880, MRR: 0.5741
test [full], NDCG@5: 0.7164, HR@5: 0.8798, NDCG@10: 0.7520, HR@10: 0.9839, MRR: 0.6805
test [sampled], NDCG@5: 0.2192, HR@5: 0.2812, NDCG@10: 0.2711, HR@10: 0.4401, MRR: 0.2470
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs

### refine_ml50_do035_s2024


In [13]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml50_do035_s2024
epoch=1, loss=0.6319
epoch=2, loss=0.2738
epoch=3, loss=0.2028
epoch=4, loss=0.1663
epoch=5, loss=0.1459
valid [full], NDCG@5: 0.7132, HR@5: 0.9064, NDCG@10: 0.7416, HR@10: 0.9930, MRR: 0.6644
valid [sampled], NDCG@5: 0.5636, HR@5: 0.5671, NDCG@10: 0.5730, HR@10: 0.5970, MRR: 0.5830
test [full], NDCG@5: 0.7275, HR@5: 0.8502, NDCG@10: 0.7759, HR@10: 1.0000, MRR: 0.7072
test [sampled], NDCG@5: 0.2035, HR@5: 0.2645, NDCG@10: 0.2580, HR@10: 0.4318, MRR: 0.2342
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-predictio

### refine_ml50_do035_s7


In [14]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml50_do035_s7
epoch=1, loss=0.5747
epoch=2, loss=0.2861
epoch=3, loss=0.2143
epoch=4, loss=0.1756
epoch=5, loss=0.1523
valid [full], NDCG@5: 0.6363, HR@5: 0.7473, NDCG@10: 0.7124, HR@10: 0.9714, MRR: 0.6369
valid [sampled], NDCG@5: 0.5617, HR@5: 0.5622, NDCG@10: 0.5638, HR@10: 0.5688, MRR: 0.5767
test [full], NDCG@5: 0.5313, HR@5: 0.6785, NDCG@10: 0.5777, HR@10: 0.8147, MRR: 0.5182
test [sampled], NDCG@5: 0.1716, HR@5: 0.1724, NDCG@10: 0.1867, HR@10: 0.2223, MRR: 0.2107
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/o

## Train only missing seeds for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### anchor_ml20_s2024


In [15]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_ml20_s2024
epoch=1, loss=0.6261
epoch=2, loss=0.3053
epoch=3, loss=0.2087
epoch=4, loss=0.1653
epoch=5, loss=0.1432
valid [full], NDCG@5: 0.6528, HR@5: 0.7732, NDCG@10: 0.7159, HR@10: 0.9707, MRR: 0.6426
valid [sampled], NDCG@5: 0.5588, HR@5: 0.5593, NDCG@10: 0.5623, HR@10: 0.5710, MRR: 0.5754
test [full], NDCG@5: 0.7080, HR@5: 0.8769, NDCG@10: 0.7503, HR@10: 1.0000, MRR: 0.6720
test [sampled], NDCG@5: 0.2013, HR@5: 0.2642, NDCG@10: 0.2488, HR@10: 0.4097, MRR: 0.2276
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/output

### anchor_ml20_s7


In [16]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_ml20_s7
epoch=1, loss=0.6517
epoch=2, loss=0.3093
epoch=3, loss=0.2163
epoch=4, loss=0.1750
epoch=5, loss=0.1489
valid [full], NDCG@5: 0.6739, HR@5: 0.8165, NDCG@10: 0.7175, HR@10: 0.9552, MRR: 0.6484
valid [sampled], NDCG@5: 0.5554, HR@5: 0.5575, NDCG@10: 0.5649, HR@10: 0.5880, MRR: 0.5741
test [full], NDCG@5: 0.7164, HR@5: 0.8798, NDCG@10: 0.7520, HR@10: 0.9839, MRR: 0.6805
test [sampled], NDCG@5: 0.2192, HR@5: 0.2812, NDCG@10: 0.2711, HR@10: 0.4401, MRR: 0.2470
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/s

### refine_ml50_do035_s2024


In [17]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml50_do035_s2024
epoch=1, loss=0.6319
epoch=2, loss=0.2738
epoch=3, loss=0.2028
epoch=4, loss=0.1663
epoch=5, loss=0.1459
valid [full], NDCG@5: 0.7132, HR@5: 0.9064, NDCG@10: 0.7416, HR@10: 0.9930, MRR: 0.6644
valid [sampled], NDCG@5: 0.5636, HR@5: 0.5671, NDCG@10: 0.5730, HR@10: 0.5970, MRR: 0.5830
test [full], NDCG@5: 0.7275, HR@5: 0.8502, NDCG@10: 0.7759, HR@10: 1.0000, MRR: 0.7072
test [sampled], NDCG@5: 0.2035, HR@5: 0.2645, NDCG@10: 0.2580, HR@10: 0.4318, MRR: 0.2342
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

### refine_ml50_do035_s7


In [18]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml50_do035_s7
epoch=1, loss=0.5747
epoch=2, loss=0.2861
epoch=3, loss=0.2143
epoch=4, loss=0.1756
epoch=5, loss=0.1523
valid [full], NDCG@5: 0.6363, HR@5: 0.7473, NDCG@10: 0.7124, HR@10: 0.9714, MRR: 0.6369
valid [sampled], NDCG@5: 0.5617, HR@5: 0.5622, NDCG@10: 0.5638, HR@10: 0.5688, MRR: 0.5767
test [full], NDCG@5: 0.5313, HR@5: 0.6785, NDCG@10: 0.5777, HR@10: 0.8147, MRR: 0.5182
test [sampled], NDCG@5: 0.1716, HR@5: 0.1724, NDCG@10: 0.1867, HR@10: 0.2223, MRR: 0.2107
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/out

## Rebuild result table from run folders

This avoids schema issues in `experiment_index.csv` and lets us combine existing and newly added runs safely.


In [19]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [20]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)


## `NDCG@10` sanity check summary


In [21]:
ndcg10_targets = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

df10 = rebuild_df(NDCG10_OUTPUT_DIR)
df10_sc = df10[df10['run_name'].isin(ndcg10_targets)].copy()
df10_sc = df10_sc.sort_values(['run_name']).reset_index(drop=True)
df10_sc[[
    'run_name', 'seed',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,anchor_ml20_s2024,2024,0.736821,0.967429,0.700301,0.850202,0.668534,0.840611,0.999865,0.819421,0.937829,0.789072,0.574314,0.616064,0.557801,0.563197,0.578608,0.323103,0.518524,0.261120,0.326822,0.292464
1,anchor_ml20_s42,42,0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.000000,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
2,anchor_ml20_s7,7,0.753294,0.961857,0.729167,0.883942,0.691225,0.913637,1.000000,0.903861,0.971918,0.885501,0.601240,0.661113,0.580044,0.594609,0.597937,0.480200,0.667438,0.420861,0.484209,0.443705
3,refine_ml50_do035_s2024,2024,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183
4,refine_ml50_do035_s42,42,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
5,refine_ml50_do035_s7,7,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750


In [22]:
df10_sc['candidate'] = df10_sc['run_name'].apply(
    lambda x: 'anchor_ml20' if 'anchor_ml20' in x else 'refine_ml50_do035'
)
summary10 = df10_sc.groupby('candidate')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary10


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                     mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
candidate                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
anchor_ml20                      0.746827  0.008788              0.968944  0.007954               0.712751  0.014836             0.860883  0.019988            0.680520  0.011400               0.889848  0.042649             0.999955  0.000078              0.879032  0.051864            0.968510  0.029126           0.853805  0.056064                   0.586415  0.013668                 0.643839  0.024291                  0.564734  0.013279                0.575209  0.016959               0.584993  0.011211                  0.422731  0.086620                0.612810  0.081997                 0.362874  0.088408               0.428098  0.087878              0.388797  0.083699
refine_ml50_do035                0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.105721              0.333054  0.093175

Interpretation guide for `NDCG@10`:
- compare mean/std of `best_valid_full_ndcg@10` and `best_test_full_ndcg@10`
- then check whether `@5`, sampled, and MRR show a similar trend


## `NDCG@5` sanity check summary


In [23]:
ndcg5_targets = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

df5 = rebuild_df(NDCG5_OUTPUT_DIR)
df5_sc = df5[df5['run_name'].isin(ndcg5_targets)].copy()
df5_sc = df5_sc.sort_values(['run_name']).reset_index(drop=True)
df5_sc[[
    'run_name', 'seed',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,anchor_ml20_s2024,2024,0.736821,0.967429,0.700301,0.850202,0.668534,0.840611,0.999865,0.819421,0.937829,0.789072,0.574314,0.616064,0.557801,0.563197,0.578608,0.323103,0.518524,0.261120,0.326822,0.292464
1,anchor_ml20_s42,42,0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.000000,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
2,anchor_ml20_s7,7,0.753294,0.961857,0.729167,0.883942,0.691225,0.913637,1.000000,0.903861,0.971918,0.885501,0.601240,0.661113,0.580044,0.594609,0.597937,0.480200,0.667438,0.420861,0.484209,0.443705
3,refine_ml50_do035_s2024,2024,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183
4,refine_ml50_do035_s42,42,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
5,refine_ml50_do035_s7,7,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750


In [24]:
df5_sc['candidate'] = df5_sc['run_name'].apply(
    lambda x: 'anchor_ml20' if 'anchor_ml20' in x else 'refine_ml50_do035'
)
summary5 = df5_sc.groupby('candidate')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary5


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                     mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
candidate                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
anchor_ml20                      0.746827  0.008788              0.968944  0.007954               0.712751  0.014836             0.860883  0.019988            0.680520  0.011400               0.889848  0.042649             0.999955  0.000078              0.879032  0.051864            0.968510  0.029126           0.853805  0.056064                   0.586415  0.013668                 0.643839  0.024291                  0.564734  0.013279                0.575209  0.016959               0.584993  0.011211                  0.422731  0.086620                0.612810  0.081997                 0.362874  0.088408               0.428098  0.087878              0.388797  0.083699
refine_ml50_do035                0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.105721              0.333054  0.093175

Interpretation guide for `NDCG@5`:
- compare mean/std of `best_valid_full_ndcg@5` and `best_test_full_ndcg@5`
- then check whether `@10`, sampled, and MRR show a similar trend
